In [1]:
import polars as pl

In [4]:
from datetime import datetime, timedelta
import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import Polygon
from tqdm import tqdm

# Assuming 'df' is your loaded 111M+ row DataFrame containing columns: [uid, d, t, x, y]

def recover_yjmob_geometries(df: pd.DataFrame, skip_polygons: bool = False) -> gpd.GeoDataFrame:
    print("Step 1: Reconstructing absolute calendar timeline...")
    # Base chronological anchor: Sunday, September 15, 2019
    base_date = datetime(2019, 9, 15)

    # Convert d (days) and t (30-minute intervals) into exact timestamps
    df["timestamp"] = base_date + pd.to_timedelta(
        df["d"], unit="D"
    ) + pd.to_timedelta(df["t"] * 30, unit="m")

    print("Step 2: Undoing geometric matrix transformations...")
    # Invert the R90(Fy(D)) transformation highlighted in Section 4.2
    # This restores proper cardinal orientation axes
    x_oriented = df["y"]
    y_oriented = -df["x"]

    # Calculate standard mean offsets to isolate grid-relative steps
    x_mean = x_oriented.mean()
    y_mean = y_oriented.mean()

    print("Step 3: Correcting aspect-ratio stretch and step scaling...")
    # Grid steps are exactly 500 meters
    STEP_SIZE = 500.0

    # Cancel out the code's 1.1x and 0.9x structural distortions
    x_metric_deltas = ((x_oriented - x_mean) / 1.1) * STEP_SIZE
    y_metric_deltas = ((y_oriented - y_mean) / 0.9) * STEP_SIZE

    print("Step 4: Mapping to Japan Plane Rectangular CS VII (EPSG:2449)...")
    # Metric coordinate values corresponding to the paper's fine-tuned WGS84 anchor
    ANCHOR_X_2449 = -11610.0
    ANCHOR_Y_2449 = -104950.0

    # Project coordinates out to exact metric bounds
    df["metric_x"] = ANCHOR_X_2449 + x_metric_deltas
    df["metric_y"] = ANCHOR_Y_2449 + y_metric_deltas

    print("Step 5 & 6: Generating and reprojecting unique bounding cells...")
    half_step = STEP_SIZE / 2.0

    # 1. Extract only the unique coordinate pairs
    unique_cells = df[["metric_x", "metric_y"]].drop_duplicates().copy()

    # 2. Construct polygons just for the unique coordinates
    unique_cells["geometry"] = [
        Polygon([
            (mx - half_step, my - half_step),
            (mx + half_step, my - half_step),
            (mx + half_step, my + half_step),
            (mx - half_step, my + half_step),
            (mx - half_step, my - half_step),
        ])
        for mx, my in tqdm(
            zip(unique_cells["metric_x"], unique_cells["metric_y"]), 
            desc="Creating Unique Polygons", 
            total=len(unique_cells)
        )
    ]

    # 3. Convert the SMALL unique dataframe to a GeoDataFrame (EPSG:2449)
    gdf_unique = gpd.GeoDataFrame(unique_cells, geometry="geometry", crs=2449)

    # 4. Reproject ONLY the unique cells to WGS84 (EPSG:4326)
    print("Reprojecting unique cells to standard WGS84 (Lat/Lon)...")
    gdf_unique = gdf_unique.to_crs(epsg=4326)

    # 5. Extract centroids while the dataset is still small
    centroids = gdf_unique.geometry.centroid
    gdf_unique["lon"] = centroids.x
    gdf_unique["lat"] = centroids.y

    print("Step 7: Merging finished WGS84 geometries back to full dataset...")
    # 6. Map the fully processed geometries and lat/lons back to the 111M rows
    df = df.merge(
        gdf_unique[["metric_x", "metric_y", "geometry", "lat", "lon"]], 
        on=["metric_x", "metric_y"], 
        how="left"
    )

    # 7. Initialize the final massive GeoDataFrame 
    # (Since it's just adopting the already-projected geometry column, this is instant)
    gdf_wgs84 = gpd.GeoDataFrame(
        df[["uid", "timestamp", "lat", "lon", "geometry", "d", "t"]], 
        geometry="geometry", 
        crs=4326
    )

    if skip_polygons:
        print("Skipping polygon generation as requested.")
        gdf_wgs84 = gdf_wgs84.drop(columns=["geometry"])

    print("Processing complete.")
    return gdf_wgs84

In [6]:
df = pl.read_csv("../../data/yjmob2/yjmob100k-dataset2.csv.gz").to_pandas()
df

,uid,d,t,x,y
0,0,0,0,163,60
1,0,0,1,163,60
2,0,0,2,163,61
3,0,0,5,163,60
4,0,0,8,163,61
...,...,...,...,...,...
29389744,24999,69,23,159,76
29389745,24999,69,24,159,69
29389746,24999,69,26,160,65
29389747,24999,69,27,160,65


In [7]:
gdf = recover_yjmob_geometries(df)
gdf

Step 1: Reconstructing absolute calendar timeline...
Step 2: Undoing geometric matrix transformations...
Step 3: Correcting aspect-ratio stretch and step scaling...
Step 4: Mapping to Japan Plane Rectangular CS VII (EPSG:2449)...
Step 5 & 6: Generating and reprojecting unique bounding cells...


Creating Unique Polygons: 100%|██████████| 30277/30277 [00:00<00:00, 155581.92it/s]
/tmp/ipykernel_711422/733253312.py:77: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = gdf_unique.geometry.centroid


Reprojecting unique cells to standard WGS84 (Lat/Lon)...
Step 7: Merging finished WGS84 geometries back to full dataset...
Processing complete.


,uid,timestamp,lat,lon,geometry,d,t
0,0,2019-09-15 00:00:00,34.853790,136.914062,"POLYGON ((136.91133 34.85153, 136.9168 34.8515...",0,0
1,0,2019-09-15 00:30:00,34.853790,136.914062,"POLYGON ((136.91133 34.85153, 136.9168 34.8515...",0,1
2,0,2019-09-15 01:00:00,34.853800,136.919033,"POLYGON ((136.91631 34.85154, 136.92177 34.851...",0,2
3,0,2019-09-15 02:30:00,34.853790,136.914062,"POLYGON ((136.91133 34.85153, 136.9168 34.8515...",0,5
4,0,2019-09-15 04:00:00,34.853800,136.919033,"POLYGON ((136.91631 34.85154, 136.92177 34.851...",0,8
...,...,...,...,...,...,...,...
29389744,24999,2019-11-23 11:30:00,34.873962,136.993554,"POLYGON ((136.99082 34.8717, 136.99629 34.8717...",69,23
29389745,24999,2019-11-23 12:00:00,34.873907,136.958749,"POLYGON ((136.95602 34.87165, 136.96149 34.871...",69,24
29389746,24999,2019-11-23 13:00:00,34.868863,136.938875,"POLYGON ((136.93615 34.8666, 136.94162 34.8666...",69,26
29389747,24999,2019-11-23 13:30:00,34.868863,136.938875,"POLYGON ((136.93615 34.8666, 136.94162 34.8666...",69,27


In [ ]:
gdf = recover_yjmob_geometries(df, skip_polygons=True)
gdf.to_parquet(
    "../../data/yjmob2/yjmob2_wgs84_simple.parquet",
    engine="pyarrow",
    compression="zstd",       # Zstandard provides excellent compression for spatial data
    index=False,              # Set to False to save space (unless your index has specific meaning)
    row_group_size=1_000_000  # Chunks the file into 1-million-row blocks for lazy loading later
)
gdf